# Mesh-to-mesh transfer in legacy FEniCS (2019.2)

This notebook gives a **clean, internally consistent** demo of how to move a finite-element function from one mesh to another in **legacy FEniCS / DOLFIN 2019.2**.

We compare several viewpoints:

1. **Algebraic transfer / prolongation** using `PETScDMCollection.create_transfer_matrix`.
2. **Target-space $L^2$ projection** of the physical source function.
3. The same $L^2$ projection written in matrix form
   \[
   M_t c_t = b_t,
   \qquad
   (M_t)_{ij} = \int_\Omega \phi_j^{(t)} \phi_i^{(t)}\,dx,
   \qquad
   (b_t)_i = \int_\Omega u_s^\sharp \phi_i^{(t)}\,dx.
   \]
4. An **oversampled least-squares / pseudoinverse** fit
   \[
   c_t^{\mathrm{ls}}
   = \arg\min_{c \in \mathbb{R}^{n_t}} \| S_t c - y \|_2^2,
   \qquad
   c_t^{\mathrm{ls}} = S_t^+ y,
   \]
   where $S_t$ samples target basis functions at many physical points.

A key point is that in legacy FEniCS one **cannot** directly assemble a UFL form containing test/trial functions from two different meshes. So we do **not** attempt to build a mixed-mesh form like
\[
\int \psi_j^{(s)} \phi_i^{(t)}\,dx
\]
directly in UFL. Instead, we either use the built-in transfer matrix, evaluate the source function physically on the target mesh and then project, or build a sampled rectangular least-squares system.

In [ ]:
from dolfin import *
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

parameters["allow_extrapolation"] = True

import dolfin
print("FEniCS version:", dolfin.__version__)

## 1. Source mesh, target mesh, and a reference function

We work with two conforming meshes of the same physical domain:
\[
\Omega = [0,1]^2.
\]

Let

- $V_s$ be the source space on the coarse mesh,
- $V_t$ be the target space on the fine mesh.

We use continuous piecewise linear elements:
\[
V_s = \mathcal{P}_1(\mathcal{T}_s)\cap H^1(\Omega),
\qquad
V_t = \mathcal{P}_1(\mathcal{T}_t)\cap H^1(\Omega).
\]

We choose a smooth exact field $u_{\mathrm{exact}}$ so that we can compare transfer errors against an exact target representation.

In [ ]:
mesh_s = UnitSquareMesh(8, 8)     # source / coarse
mesh_t = UnitSquareMesh(24, 24)   # target / fine

V_s = FunctionSpace(mesh_s, "CG", 1)
V_t = FunctionSpace(mesh_t, "CG", 1)

u_exact_expr = Expression(
    "sin(2*pi*x[0])*cos(3*pi*x[1]) + 0.2*cos(5*pi*x[0]*x[1]) + x[0]*(1.0 - x[1])",
    degree=6, pi=np.pi
)

u_s = interpolate(u_exact_expr, V_s)
u_t_exact = interpolate(u_exact_expr, V_t)

print("dim(V_s) =", V_s.dim())
print("dim(V_t) =", V_t.dim())

## 2. A few utilities

We will measure errors in coefficient space and in physical norms.

For two target-space functions $u,v \in V_t$ we use the relative $L^2$ error
\[
\frac{\|u-v\|_{L^2(\Omega)}}{\|v\|_{L^2(\Omega)}}.
\]

When comparing coefficient vectors $c,\hat c \in \mathbb{R}^{n_t}$ we use the Euclidean relative error
\[
\frac{\|c-\hat c\|_2}{\|\hat c\|_2}.
\]

In [ ]:
def rel_l2_error(u_num, u_ref):
    return errornorm(u_ref, u_num, norm_type="L2") / norm(u_ref, "L2")

def rel_h1_error(u_num, u_ref):
    return errornorm(u_ref, u_num, norm_type="H1") / norm(u_ref, "H1")

def coeffs(f):
    return f.vector().get_local().copy()

def rel_vec_error(a, b):
    return np.linalg.norm(a - b) / np.linalg.norm(b)

def report(name, u_num, u_ref=u_t_exact):
    print(f"{name:35s}  rel L2 = {rel_l2_error(u_num, u_ref):.6e}   rel H1 = {rel_h1_error(u_num, u_ref):.6e}")

## 3. Method A: built-in algebraic transfer (prolongation)

The most direct built-in transfer is the algebraic operator
\[
P_{s\to t} \in \mathbb{R}^{n_t \times n_s},
\qquad
d_t^{\mathrm{alg}} = P_{s\to t} d_s,
\]
where $d_s$ are the source coefficients and $d_t^{\mathrm{alg}}$ are target coefficients.

In [ ]:
P_st = PETScDMCollection.create_transfer_matrix(V_s, V_t)

u_t_alg = Function(V_t)
u_t_alg.vector()[:] = P_st * u_s.vector()

report("algebraic transfer", u_t_alg)

print("P_st shape =", P_st.size(0), "x", P_st.size(1))

### Round-trip test

We can also transfer back to the coarse mesh:
\[
P_{t\to s} \in \mathbb{R}^{n_s \times n_t},
\qquad
d_s^{\mathrm{rt}} = P_{t\to s} P_{s\to t} d_s.
\]

In [ ]:
P_ts = PETScDMCollection.create_transfer_matrix(V_t, V_s)

u_s_roundtrip = Function(V_s)
u_s_roundtrip.vector()[:] = P_ts * u_t_alg.vector()

u_s_ref = interpolate(u_exact_expr, V_s)

print("round-trip coarse-space rel L2 error =", rel_l2_error(u_s_roundtrip, u_s_ref))
print("round-trip coefficient rel error     =", rel_vec_error(coeffs(u_s_roundtrip), coeffs(u_s_ref)))

## 4. Method B: target-space $L^2$ projection of the physical source function

We regard the source finite-element function $u_s$ as a physical function
\[
u_s^\sharp : \Omega \to \mathbb{R}.
\]

The target $L^2$ projection seeks $u_t^{L^2}\in V_t$ such that
\[
\int_\Omega u_t^{L^2} v_t\,dx
=
\int_\Omega u_s^\sharp v_t\,dx
\qquad \forall v_t \in V_t.
\]

In coordinates,
\[
M_t c_t = b_t,
\]
with
\[
(M_t)_{ij} = \int_\Omega \phi_j^{(t)}\phi_i^{(t)}\,dx,
\qquad
(b_t)_i = \int_\Omega u_s^\sharp \phi_i^{(t)}\,dx.
\]

A simple way to implement this in legacy FEniCS is to wrap the source function in a target-side `UserExpression`.

In [ ]:
class SourceFunctionExpression(UserExpression):
    def __init__(self, u_source, **kwargs):
        super().__init__(**kwargs)
        self.u_source = u_source

    def eval(self, values, x):
        values[0] = self.u_source(x)

    def value_shape(self):
        return ()

u_s_on_target_expr = SourceFunctionExpression(u_s, degree=6)
u_t_l2 = project(u_s_on_target_expr, V_t)

report("target L2 projection", u_t_l2)

## 5. Method C: the same $L^2$ projection, assembled explicitly as a matrix solve

Let $\{\phi_i^{(t)}\}_{i=1}^{n_t}$ denote the target basis. Then
\[
M_t = \left[\int_\Omega \phi_j^{(t)} \phi_i^{(t)}\,dx \right]_{i,j=1}^{n_t},
\qquad
b_t = \left[\int_\Omega u_s^\sharp \phi_i^{(t)}\,dx \right]_{i=1}^{n_t}.
\]

We solve
\[
M_t c_t = b_t
\]
directly in SciPy and reconstruct the target function.

This is mathematically **the same problem** as the `project(...)` section above.

In [ ]:
phi = TrialFunction(V_t)
v = TestFunction(V_t)

M_t = assemble(phi * v * dx).array()
b_t = assemble(u_s_on_target_expr * v * dx).get_local()

c_t_manual = la.solve(M_t, b_t, assume_a="sym")

u_t_manual = Function(V_t)
u_t_manual.vector().set_local(c_t_manual)
u_t_manual.vector().apply("insert")

report("manual mass-matrix solve", u_t_manual)

print("difference between project and manual coefficients =",
      np.linalg.norm(coeffs(u_t_l2) - coeffs(u_t_manual)))

## 6. Method D: an oversampled least-squares / pseudoinverse fit

Choose sample points $\{x_q\}_{q=1}^m \subset \Omega$ with $m > n_t$, and define
\[
(S_t)_{q i} = \phi_i^{(t)}(x_q),
\qquad
y_q = u_s^\sharp(x_q).
\]

Then we solve
\[
c_t^{\mathrm{ls}}
= \arg\min_{c \in \mathbb{R}^{n_t}} \|S_t c - y\|_2^2,
\qquad
c_t^{\mathrm{ls}} = S_t^+ y.
\]

This is a clean rectangular-matrix viewpoint. It is **not** exactly the continuous $L^2$ projection unless the sampling rule is chosen to mimic the $L^2$ inner product.

In [ ]:
nx_samp = 41
xx = np.linspace(0.0, 1.0, nx_samp)
yy = np.linspace(0.0, 1.0, nx_samp)
X = np.array([[x, y] for y in yy for x in xx])

m = X.shape[0]
n_t = V_t.dim()

print("number of sample points =", m)
print("number of target dofs   =", n_t)

S_t = np.zeros((m, n_t))

for j in range(n_t):
    e_j = Function(V_t)
    vec = np.zeros(n_t)
    vec[j] = 1.0
    e_j.vector().set_local(vec)
    e_j.vector().apply("insert")
    S_t[:, j] = np.array([e_j(Point(x, y)) for x, y in X])

y = np.array([u_s(Point(x, y)) for x, y in X])

c_t_pinv = la.pinv(S_t) @ y

u_t_pinv = Function(V_t)
u_t_pinv.vector().set_local(c_t_pinv)
u_t_pinv.vector().apply("insert")

report("oversampled pinv fit", u_t_pinv)
print("least-squares residual ||S_t c - y||_2 =", np.linalg.norm(S_t @ c_t_pinv - y))
print("distance to L2-proj coefficients       =", np.linalg.norm(c_t_pinv - coeffs(u_t_manual)))

## 7. Optional weighted least-squares viewpoint

To better mimic an $L^2$ inner product, introduce weights:
\[
c_t^{W}
=
\arg\min_c \|W^{1/2}(S_t c - y)\|_2^2.
\]

For uniform tensor-product sampling on $[0,1]^2$, a simple crude choice is
\[
w_q = \Delta x \, \Delta y.
\]

Then
\[
c_t^{W}
=
\arg\min_c \sum_{q=1}^m w_q \, |\phi_c(x_q) - u_s^\sharp(x_q)|^2.
\]

In [ ]:
dx_samp = 1.0 / (nx_samp - 1)
dy_samp = 1.0 / (nx_samp - 1)
w = np.full(m, dx_samp * dy_samp)
Wsqrt = np.sqrt(w)

S_w = S_t * Wsqrt[:, None]
y_w = y * Wsqrt

c_t_weighted = la.lstsq(S_w, y_w)[0]

u_t_weighted = Function(V_t)
u_t_weighted.vector().set_local(c_t_weighted)
u_t_weighted.vector().apply("insert")

report("weighted least-squares fit", u_t_weighted)
print("distance to L2-proj coefficients =", np.linalg.norm(c_t_weighted - coeffs(u_t_manual)))

## 8. Summary table

We summarize the transfer errors against the exact target-space representation
\[
u_t^\star = \Pi_{V_t} u_{\mathrm{exact}}.
\]

Interpretation:

- **Algebraic transfer** is the direct built-in prolongation.
- **$L^2$ projection** is variationally natural.
- **Manual mass solve** should match `project`.
- **Pseudoinverse / least-squares** gives a rectangular-matrix story, but it solves a sampled problem.

In [ ]:
methods = [
    ("algebraic transfer", u_t_alg),
    ("target L2 projection", u_t_l2),
    ("manual mass-matrix solve", u_t_manual),
    ("oversampled pinv fit", u_t_pinv),
    ("weighted least-squares fit", u_t_weighted),
]

print(f"{'method':35s} {'rel L2':>14s} {'rel H1':>14s}")
print("-" * 65)
for name, ufun in methods:
    print(f"{name:35s} {rel_l2_error(ufun, u_t_exact):14.6e} {rel_h1_error(ufun, u_t_exact):14.6e}")

## 9. Visual comparison

In [ ]:
plt.figure(figsize=(14, 10))

plt.subplot(2, 3, 1)
p = plot(u_t_exact)
plt.colorbar(p)
plt.title("exact target reference")

plt.subplot(2, 3, 2)
p = plot(u_t_alg)
plt.colorbar(p)
plt.title("algebraic transfer")

plt.subplot(2, 3, 3)
p = plot(u_t_l2)
plt.colorbar(p)
plt.title("target $L^2$ projection")

plt.subplot(2, 3, 4)
p = plot(u_t_manual)
plt.colorbar(p)
plt.title("manual mass solve")

plt.subplot(2, 3, 5)
p = plot(u_t_pinv)
plt.colorbar(p)
plt.title("oversampled pinv fit")

plt.subplot(2, 3, 6)
p = plot(u_t_weighted)
plt.colorbar(p)
plt.title("weighted LS fit")

plt.tight_layout()
plt.show()

## 10. Remarks

### Why there is no mixed-mesh UFL Gram matrix here

It is tempting to try to write a cross-mesh matrix
\[
(B_{ts})_{ij} = \int_\Omega \psi_j^{(s)} \phi_i^{(t)}\,dx.
\]
However, in **legacy FEniCS**, if $\psi_j^{(s)}$ and $\phi_i^{(t)}$ belong to spaces on different meshes, then a direct UFL form containing both of them is not supported in the way one might hope.

So this notebook keeps the mathematics aligned with what the code actually does:

- same-mesh variational forms for target-space projection,
- algebraic prolongation via built-in transfer matrices,
- rectangular sampled matrices for pseudoinverse least-squares.

### If you want a true cross-Gram matrix

Then the correct route is to assemble it **manually** by quadrature and point evaluation:
\[
(B_{ts})_{ij}
\approx
\sum_{q=1}^{N_q} w_q \, \psi_j^{(s)}(x_q) \phi_i^{(t)}(x_q),
\]
with quadrature points $x_q$ chosen on the physical domain and both source and target basis functions evaluated there.

That is doable, but it is a more elaborate custom implementation than the clean built-in FEniCS demo shown here.